# 01. BaseModel の基本と型変換（Lax vs. Strict モード）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuichi110/drawlib-examples/blob/main/docs/pydantic/ja/01_basemodel_quickstart.ipynb)

Pydantic v2 は、Python 標準の型ヒント（`class User(BaseModel):`）を Rust 製の高速コアエンジン `pydantic-core` による型検証・シリアライズパイプラインへと変換します。`pydantic` v2 は Google Colab に標準でプリインストールされており、API キーや外部サービス設定も一切不要なため、本クックブックのすべてのコードセルをそのまま即座に実行できます。

![図1-1: Pydantic v2 における型検証・変換・シリアライズの一貫パイプライン](https://examples.drawlib.com/pydantic/images/01_basemodel_quickstart/ch01_pydantic_pipeline_ja.png)

*図1-1: Pydantic v2 における型検証・変換・シリアライズの一貫パイプライン*

## 1. 最初の `BaseModel` と自動型変換（Lax モード）

Pydantic v2 でデータスキーマを定義するには、`BaseModel` を継承して Python 標準の型アノテーション（`int`, `str`, `bool`, `list[str]`）を記述するだけです。既定の **Lax モード（柔軟モード）** では、URL クエリ文字列やフォーム送信、外部 JSON ペイロードから `"42"` や `"true"` といった文字列が渡されても、`pydantic-core` が互換性のある値をネイティブの `int` や `bool` へ安全に変換し、実行時の型安全性を保証します。

In [ ]:
!pip install -q pydantic

In [ ]:
from pydantic import BaseModel


class User(BaseModel):
    id: int
    name: str
    is_active: bool = True
    tags: list[str] = []


user = User(id="42", name="Alice", is_active="true", tags=["admin", "colab"])

print(user)
print("type(user.id):", type(user.id), "| is_active:", user.is_active)
print("model_dump():", user.model_dump())

また Pydantic v2 では、旧 v1 のメソッド名（`parse_obj`, `parse_raw`, `dict`, `json`）が非推奨となり、`model_*` プレフィックスの統一メソッドへ刷新されています。

- **`Model.model_validate(obj)`** *(旧 v1: `parse_obj`)*: Python の `dict` やオブジェクトを検証し、型付きの `BaseModel` インスタンスを生成します。
- **`Model.model_validate_json(data)`** *(旧 v1: `parse_raw`)*: 生の UTF-8 JSON 文字列（または `bytes`）を Python の `json.loads()` を介さず Rust 内部（`jiter`）で一括パース＆検証するため、高速に動作します。
- **`instance.model_dump()` / `instance.model_dump_json()`** *(旧 v1: `dict()` / `json()`)*: 検証済みモデルを Python の `dict` または整形済み JSON 文字列へシリアライズします。

In [ ]:
raw_json = '{"id": "108", "name": "Bob", "tags": ["data", "rust"]}'

bob = User.model_validate_json(raw_json)
print("Validated from JSON:", bob)
print(bob.model_dump_json(indent=2))

## 2. Lax モードと Strict モードの使い分け（`ConfigDict(strict=True)`）

外部入力を受け取る境界層（HTTP クエリパラメータ、環境変数、CSV 取り込みなど）では自動型変換を行う **Lax モード** が便利ですが、マイクロサービス間の内部契約や決済・金融データ処理では、`"42"` を暗黙的に `42` へ変換せず型不一致バグとして即座に弾きたい場面があります。

Pydantic v2 では、**呼び出し単位**（`strict=True` 引数）または **モデル全体**（`model_config = ConfigDict(strict=True)`）で **Strict モード（厳格モード）** を有効化できます。

- **Lax モード（既定）**: `"42"` → `42`、`"true"` → `True`、`[1, 2]` → `(1, 2)` のように解釈可能な入力を自動変換しつつ、`"forty-two"` のような変換不能な値は確実に拒否します。
- **Strict モード（`strict=True`）**: Python オブジェクト検証時に厳密な型一致を要求します（`int` に対する `"42"` は `int_type`、`bool` に対する `"true"` は `bool_type`、`tuple[int, int]` に対する `[1, 2]` は `tuple_type` エラーとなります）。

![図1-2: Lax モード（既定の自動型変換）と Strict モード（厳格型一致）の挙動比較](https://examples.drawlib.com/pydantic/images/01_basemodel_quickstart/ch01_lax_vs_strict_ja.png)

*図1-2: Lax モード（既定の自動型変換）と Strict モード（厳格型一致）の挙動比較*

先ほど定義した `User` モデルを使って、既定の Lax モードと呼び出し単位の `strict=True`、さらに `ConfigDict(strict=True)` によるモデル単位の厳格検証を実際に比較してみましょう。

In [ ]:
from pydantic import ConfigDict, ValidationError

payload = {"id": "42", "name": "Alice"}

# 1. 既定の Lax モード: 文字列 "42" を int の 42 へ自動変換
lax_user = User.model_validate(payload)
print("Lax mode succeeded:", repr(lax_user))

# 2. 呼び出し単位の Strict モード: int に対する文字列 "42" を拒否
try:
    User.model_validate(payload, strict=True)
except ValidationError as e:
    print("\nStrict mode raised ValidationError:")
    print(e)


# 3. ConfigDict(strict=True) によるモデル全体の Strict モード化
class StrictPoint(BaseModel):
    model_config = ConfigDict(strict=True)

    label: str
    coords: tuple[int, int]


point = StrictPoint(label="origin", coords=(0, 0))
print("\nStrictPoint exact match:", repr(point))

`ValidationError` の出力には、違反したフィールド名（`id`）、分かりやすいエラーメッセージ（`Input should be a valid integer`）、エラー種別コード（`type=int_type`）、実際に入力された値（`input_value='42'`）が正確に記録されます。なお JSON にはタプル構文が存在しないため、`StrictPoint.model_validate_json('{"label": "origin", "coords": [0, 0]}')` のように JSON 文字列を直接検証する場合は、Strict モードでも JSON 配列 `[0, 0]` が `tuple[int, int]` として受理されます。

### まとめと次のステップ

- **`BaseModel` と v2 標準 `model_*` メソッド**: Python 標準の型ヒントを宣言するだけで、`model_validate()` / `model_validate_json()` による Rust 高速検証と `model_dump()` / `model_dump_json()` による出力を一貫して扱えます。
- **次章（Chapter 02）のプレビュー**: 続く第2章では、`Field(...)` による数値範囲・文字列パターン制約や安全な `default_factory` の定義方法と、複数フィールドにわたる `ValidationError`（`e.errors()`）の構造的な解析手法を学びます。